# Baseline Anomaly Detection

In this notebook, we train baseline anomaly detection models using the preprocessed CICIDS2017 network-flow data.

The model is trained only on the Monday BENIGN traffic, which represents normal network behavior. It is then evaluated on the Friday afternoon DDoS file, which contains both BENIGN and DDoS traffic.

The goal is to check whether attack flows receive higher anomaly scores than normal flows.

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

print("Project root:", Path.cwd())

Project root: D:\Development\Python Projects\network-traffic-anomaly-detection


In [2]:
import pandas as pd
import numpy as np

In [3]:
processed_dir = Path("data/processed")

X_train = pd.read_csv(processed_dir / "X_train_scaled.csv")
X_test = pd.read_csv(processed_dir / "X_test_scaled.csv")

y_test = pd.read_csv(processed_dir / "y_test_binary.csv").squeeze()
y_test_labels = pd.read_csv(processed_dir / "y_test_labels.csv").squeeze()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nTest label distribution:")
print(y_test.value_counts())

X_train shape: (502650, 78)
X_test shape: (223082, 78)
y_test shape: (223082,)

Test label distribution:
Label
1    128014
0     95068
Name: count, dtype: int64


## Isolation Forest Baseline

Isolation Forest is used as the first anomaly detection baseline. It works by randomly partitioning the feature space. Anomalous samples are expected to be easier to isolate than normal samples, so they tend to have shorter average path lengths in the isolation trees.

The model is trained only on normal BENIGN traffic. During evaluation, it assigns anomaly scores to both BENIGN and DDoS samples in the test set.

In [5]:
from sklearn.ensemble import IsolationForest

iso_forest = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_train)



IsolationForest(n_jobs=-1, random_state=42)

In [6]:
iso_scores = -iso_forest.decision_function(X_test)

print("Anomaly scores shape:", iso_scores.shape)
print("Min score:", iso_scores.min())
print("Max score:", iso_scores.max())
print("Mean score:", iso_scores.mean())

Anomaly scores shape: (223082,)
Min score: -0.17867838471693676
Max score: 0.27267371440192634
Mean score: -0.031333380132419646


In [7]:
from sklearn.metrics import roc_auc_score, average_precision_score

roc_auc = roc_auc_score(y_test, iso_scores)
pr_auc = average_precision_score(y_test, iso_scores)

print("Isolation Forest ROC-AUC:", roc_auc)
print("Isolation Forest PR-AUC:", pr_auc)

Isolation Forest ROC-AUC: 0.6808983013346402
Isolation Forest PR-AUC: 0.6495483878417078


In [8]:
score_df = pd.DataFrame({
    "label": y_test_labels,
    "binary_label": y_test,
    "anomaly_score": iso_scores
})

score_df.groupby("label")["anomaly_score"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
BENIGN,95068.0,-0.066286,0.118334,-0.178678,-0.165348,-0.112377,-0.019246,0.272674
DDoS,128014.0,-0.005376,0.102728,-0.165953,-0.115220,0.023736,0.036469,0.186137


## Isolation Forest Result Interpretation

The Isolation Forest baseline achieved a ROC-AUC of approximately 0.68 and a PR-AUC of approximately 0.65. This indicates that the model performs better than random guessing, but the separation between BENIGN and DDoS traffic is only moderate.

The anomaly score distribution shows that DDoS samples receive higher anomaly scores on average than BENIGN samples. This suggests that the model has learned some characteristics of normal Monday traffic and can identify part of the DDoS traffic as abnormal.

However, the score ranges of BENIGN and DDoS traffic overlap. Some BENIGN flows are assigned high anomaly scores, while some DDoS flows are assigned lower anomaly scores. This overlap limits the detection performance of the Isolation Forest baseline.

Overall, Isolation Forest provides a useful first baseline, but further models and feature analysis are needed to improve anomaly detection performance.

## Threshold based evaluation

In [9]:
# Compute anomaly scores for training data
train_scores = -iso_forest.decision_function(X_train)

# Set threshold at 95th percentile of training scores
threshold = np.percentile(train_scores, 95)

print("Threshold:", threshold)

Threshold: 0.0494141663787212


In [10]:
# Predict anomaly if score is above threshold
y_pred = (iso_scores > threshold).astype(int)

print(pd.Series(y_pred).value_counts())

0    179787
1     43295
Name: count, dtype: int64


In [11]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["BENIGN", "DDoS"]))

Confusion Matrix:
[[ 74751  20317]
 [105036  22978]]

Classification Report:
              precision    recall  f1-score   support

      BENIGN       0.42      0.79      0.54     95068
        DDoS       0.53      0.18      0.27    128014

    accuracy                           0.44    223082
   macro avg       0.47      0.48      0.41    223082
weighted avg       0.48      0.44      0.39    223082



In [12]:
tn, fp, fn, tp = cm.ravel()

false_positive_rate = fp / (fp + tn)
detection_rate = tp / (tp + fn)

print("False Positive Rate:", false_positive_rate)
print("Detection Rate / Recall:", detection_rate)

False Positive Rate: 0.2137101863928977
Detection Rate / Recall: 0.17949599262580654


## Threshold-Based Isolation Forest Evaluation

Using the 95th percentile of the training anomaly scores as the decision threshold, the Isolation Forest model achieved a low DDoS recall of approximately 0.18. This means that only about 18% of DDoS flows were detected as anomalies.

The false positive rate was approximately 0.21, meaning that around 21% of BENIGN test flows were incorrectly flagged as anomalous.

The confusion matrix shows that the main weakness of this threshold is the high number of false negatives. Many DDoS samples receive anomaly scores below the selected threshold and are therefore classified as normal.

This suggests that although Isolation Forest provides some ranking separation between BENIGN and DDoS traffic, the 95th percentile threshold is too conservative for this experiment. Further threshold analysis is required to study the trade-off between detection rate and false positive rate.

In [13]:
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

for percentile in [50, 60, 70, 80, 85, 90, 95, 97, 99]:
    threshold = np.percentile(train_scores, percentile)
    y_pred = (iso_scores > threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    fpr = fp / (fp + tn)

    threshold_results.append({
        "percentile": percentile,
        "threshold": threshold,
        "precision": precision,
        "recall_detection_rate": recall,
        "f1_score": f1,
        "false_positive_rate": fpr,
        "predicted_anomalies": y_pred.sum()
    })

threshold_results_df = pd.DataFrame(threshold_results)
threshold_results_df

,percentile,threshold,precision,recall_detection_rate,f1_score,false_positive_rate,predicted_anomalies
0,50,-0.145327,0.683457,0.974909,0.803572,0.608007,182604
1,60,-0.132860,0.672574,0.854133,0.752558,0.559915,162571
2,70,-0.114570,0.660326,0.732959,0.694749,0.507700,142095
3,80,-0.085588,0.651059,0.640211,0.645590,0.462038,125881
4,85,-0.060577,0.696089,0.636368,0.664890,0.374122,117031
5,90,-0.027680,0.770613,0.636071,0.696908,0.254954,105664
6,95,0.049414,0.530731,0.179496,0.268264,0.213710,43295
7,97,0.079955,0.654104,0.168255,0.267660,0.119809,32929
8,99,0.132160,0.692262,0.164443,0.265757,0.098435,30409


## Threshold Comparison Interpretation

The threshold comparison shows a clear trade-off between detection rate and false positive rate. Lower thresholds detect more DDoS flows, but they also incorrectly flag a large portion of BENIGN traffic as anomalous. Higher thresholds reduce false positives but miss many DDoS samples.

The 50th percentile threshold achieves very high DDoS recall, but it also produces a false positive rate of approximately 61%, which would be too noisy for a practical monitoring system.

The 90th percentile threshold provides a more balanced operating point. It achieves approximately 77% precision, 64% recall, and an F1-score of around 0.70, while keeping the false positive rate lower than the more aggressive thresholds.

Therefore, the 90th percentile threshold is selected as the practical Isolation Forest baseline threshold for this experiment.

In [14]:
selected_percentile = 90
selected_threshold = np.percentile(train_scores, selected_percentile)

y_pred_iso = (iso_scores > selected_threshold).astype(int)

print("Selected percentile:", selected_percentile)
print("Selected threshold:", selected_threshold)
print("Predicted anomalies:", y_pred_iso.sum())

Selected percentile: 90
Selected threshold: -0.027679601296038792
Predicted anomalies: 105664


In [15]:
from sklearn.metrics import confusion_matrix, classification_report

cm_iso = confusion_matrix(y_test, y_pred_iso)

print("Confusion Matrix:")
print(cm_iso)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_iso, target_names=["BENIGN", "DDoS"]))

Confusion Matrix:
[[70830 24238]
 [46588 81426]]

Classification Report:
              precision    recall  f1-score   support

      BENIGN       0.60      0.75      0.67     95068
        DDoS       0.77      0.64      0.70    128014

    accuracy                           0.68    223082
   macro avg       0.69      0.69      0.68    223082
weighted avg       0.70      0.68      0.68    223082



In [16]:
tn, fp, fn, tp = cm_iso.ravel()

fpr_iso = fp / (fp + tn)
recall_iso = tp / (tp + fn)

print("False Positive Rate:", fpr_iso)
print("Detection Rate / Recall:", recall_iso)

False Positive Rate: 0.25495434846636095
Detection Rate / Recall: 0.6360710547283891


In [17]:
iso_summary = pd.DataFrame([{
    "model": "Isolation Forest",
    "threshold_percentile": selected_percentile,
    "roc_auc": roc_auc,
    "pr_auc": pr_auc,
    "precision": 0.770613,
    "recall": 0.636071,
    "f1_score": 0.696908,
    "false_positive_rate": 0.254954
}])

iso_summary

,model,threshold_percentile,roc_auc,pr_auc,precision,recall,f1_score,false_positive_rate
0,Isolation Forest,90,0.680898,0.649548,0.770613,0.636071,0.696908,0.254954


## Isolation Forest Baseline Summary

Isolation Forest was trained only on the Monday BENIGN traffic and evaluated on the Friday afternoon BENIGN + DDoS traffic. The model achieved a ROC-AUC of approximately 0.68 and a PR-AUC of approximately 0.65, showing moderate ability to rank DDoS flows as more anomalous than BENIGN flows.

A threshold comparison was performed using different percentiles of the training anomaly score distribution. Lower thresholds increased DDoS detection but caused high false positive rates. Higher thresholds reduced false positives but missed many attacks.

The 90th percentile threshold was selected as the most practical operating point. At this threshold, the model achieved approximately 77% precision, 64% recall, and an F1-score of about 0.70, with a false positive rate of about 25%.

This result provides a useful first baseline, but the false positive rate is still high for a real monitoring system. Therefore, additional anomaly detection methods should be evaluated and compared.

## Local Outlier Factor Baseline

Local Outlier Factor (LOF) is used as a second unsupervised anomaly detection baseline. LOF compares the local density of a sample with the local density of its neighbors. Samples that exist in lower-density regions compared to their neighbors are considered more anomalous.

Since LOF can be computationally expensive on large datasets, a random sample of the normal training data is used for fitting the model. The model is trained only on BENIGN traffic and evaluated on the mixed BENIGN + DDoS test set.

In [18]:
X_train_sample = X_train.sample(n=50000, random_state=42)

print("Training sample shape:", X_train_sample.shape)
print("Test shape:", X_test.shape)

Training sample shape: (50000, 78)
Test shape: (223082, 78)


In [19]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(
    n_neighbors=35,
    contamination="auto",
    novelty=True,
    n_jobs=-1
)

lof.fit(X_train_sample)

LocalOutlierFactor(n_jobs=-1, n_neighbors=35, novelty=True)

In [20]:
lof_scores = -lof.decision_function(X_test)

print("LOF anomaly scores shape:", lof_scores.shape)
print("Min score:", lof_scores.min())
print("Max score:", lof_scores.max())
print("Mean score:", lof_scores.mean())

D:\Development\Python\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


LOF anomaly scores shape: (223082,)
Min score: -0.5834982538967465
Max score: 2153.870339422869
Mean score: 2.9894101975115235


In [21]:
from sklearn.metrics import roc_auc_score, average_precision_score

lof_roc_auc = roc_auc_score(y_test, lof_scores)
lof_pr_auc = average_precision_score(y_test, lof_scores)

print("LOF ROC-AUC:", lof_roc_auc)
print("LOF PR-AUC:", lof_pr_auc)

LOF ROC-AUC: 0.8026525668601054
LOF PR-AUC: 0.8115939957116551


In [22]:
lof_score_df = pd.DataFrame({
    "label": y_test_labels,
    "binary_label": y_test,
    "anomaly_score": lof_scores
})

lof_score_df.groupby("label")["anomaly_score"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
BENIGN,95068.0,1.666503,25.287750,-0.583498,-0.462563,-0.177787,1.427952,2153.870339
DDoS,128014.0,3.971851,4.760873,-0.465909,0.482417,2.354914,5.652478,89.148984


## Local Outlier Factor Result Interpretation

Local Outlier Factor achieved a ROC-AUC of approximately 0.80 and a PR-AUC of approximately 0.81. This is a clear improvement over the Isolation Forest baseline.

The score distribution shows that DDoS flows receive higher anomaly scores than BENIGN flows on average. The median DDoS anomaly score is also much higher than the median BENIGN anomaly score, indicating better separation between normal and attack traffic.

However, a few BENIGN samples receive very high anomaly scores, which suggests that some normal flows are also locally unusual. This is important in security monitoring because such cases can contribute to false positives.

Overall, LOF provides a stronger baseline than Isolation Forest for the first DDoS anomaly detection experiment.

In [23]:
lof_train_scores = -lof.decision_function(X_train_sample)

lof_threshold_results = []

for percentile in [50, 60, 70, 80, 85, 90, 95, 97, 99]:
    threshold = np.percentile(lof_train_scores, percentile)
    y_pred = (lof_scores > threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    fpr = fp / (fp + tn)

    lof_threshold_results.append({
        "percentile": percentile,
        "threshold": threshold,
        "precision": precision,
        "recall_detection_rate": recall,
        "f1_score": f1,
        "false_positive_rate": fpr,
        "predicted_anomalies": y_pred.sum()
    })

lof_threshold_results_df = pd.DataFrame(lof_threshold_results)
lof_threshold_results_df

D:\Development\Python\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


,percentile,threshold,precision,recall_detection_rate,f1_score,false_positive_rate,predicted_anomalies
0,50,-0.448990,0.652694,0.999945,0.789838,0.716477,196121
1,60,-0.405168,0.674817,0.999922,0.805814,0.648830,189687
2,70,-0.326343,0.700366,0.999680,0.823674,0.575904,182723
3,80,-0.170865,0.729894,0.999008,0.843507,0.497812,175213
4,85,-0.002331,0.730424,0.921985,0.815101,0.458198,161587
5,90,0.369867,0.729991,0.763932,0.746576,0.380486,133966
6,95,1.750611,0.789067,0.573687,0.664357,0.206505,93072
7,97,4.336478,0.911783,0.319403,0.473082,0.041612,44844
8,99,20.561000,0.239583,0.002156,0.004274,0.009214,1152


## LOF Threshold Comparison Interpretation

The LOF threshold comparison shows a clear trade-off between detection rate and false positive rate. Lower thresholds detect almost all DDoS flows but produce a very high number of false positives. For example, the 80th percentile threshold detects nearly all DDoS samples, but it incorrectly flags almost half of the BENIGN traffic as anomalous.

The 90th percentile threshold gives a strong recall and F1-score, but the false positive rate remains high at approximately 38%. For a practical monitoring system, this would likely generate too many alerts.

The 95th percentile threshold provides a more balanced operating point. It achieves approximately 79% precision, 57% recall, and a false positive rate of about 21%. This makes it a reasonable practical baseline threshold for LOF.

The 97th percentile threshold can be interpreted as a high-confidence alerting mode because it achieves high precision and a low false positive rate, but it misses a larger portion of attacks.

For the baseline comparison, the 95th percentile threshold is selected as the practical LOF operating point.

In [24]:
baseline_comparison = pd.DataFrame([
    {
        "model": "Isolation Forest",
        "selected_threshold_percentile": 90,
        "roc_auc": 0.6808983013346402,
        "pr_auc": 0.6495483878417078,
        "precision": 0.770613,
        "recall": 0.636071,
        "f1_score": 0.696908,
        "false_positive_rate": 0.254954
    },
    {
        "model": "Local Outlier Factor",
        "selected_threshold_percentile": 95,
        "roc_auc": 0.8026525668601054,
        "pr_auc": 0.8115939957116551,
        "precision": 0.789067,
        "recall": 0.573687,
        "f1_score": 0.664357,
        "false_positive_rate": 0.206505
    }
])

baseline_comparison

,model,selected_threshold_percentile,roc_auc,pr_auc,precision,recall,f1_score,false_positive_rate
0,Isolation Forest,90,0.680898,0.649548,0.770613,0.636071,0.696908,0.254954
1,Local Outlier Factor,95,0.802653,0.811594,0.789067,0.573687,0.664357,0.206505


## DBSCAN Baseline

DBSCAN is evaluated as a density-based unsupervised anomaly detection baseline. DBSCAN groups dense regions of samples into clusters and marks points that do not belong to any dense region as noise. These noise points can be interpreted as potential anomalies.

Since DBSCAN is computationally expensive on large high-dimensional datasets, this experiment is performed on a sampled subset of the training and test data. This makes the method suitable for exploratory comparison while avoiding excessive memory and runtime costs.

In [25]:
# Sample normal training data
X_train_dbscan = X_train.sample(n=30000, random_state=42)

# Combine X_test and labels for sampling
test_for_sample = X_test.copy()
test_for_sample["binary_label"] = y_test.values
test_for_sample["original_label"] = y_test_labels.values

# Sample benign and DDoS from test data
benign_sample = test_for_sample[test_for_sample["binary_label"] == 0].sample(
    n=10000, random_state=42
)

ddos_sample = test_for_sample[test_for_sample["binary_label"] == 1].sample(
    n=10000, random_state=42
)

test_sample = pd.concat([benign_sample, ddos_sample], axis=0).sample(
    frac=1, random_state=42
)

X_test_dbscan = test_sample.drop(columns=["binary_label", "original_label"])
y_test_dbscan = test_sample["binary_label"]
y_test_labels_dbscan = test_sample["original_label"]

print("DBSCAN train shape:", X_train_dbscan.shape)
print("DBSCAN test shape:", X_test_dbscan.shape)
print(y_test_dbscan.value_counts())

DBSCAN train shape: (30000, 78)
DBSCAN test shape: (20000, 78)
binary_label
1    10000
0    10000
Name: count, dtype: int64


In [26]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(
    eps=3.0,
    min_samples=10,
    n_jobs=-1
)

dbscan.fit(X_train_dbscan)

DBSCAN(eps=3.0, min_samples=10, n_jobs=-1)

In [27]:
train_cluster_labels = dbscan.labels_

pd.Series(train_cluster_labels).value_counts().sort_index()

-1      1239
 0     22707
 1      3423
 2       335
 3        25
 4       209
 5      1504
 6        89
 7        52
 8        53
 9        20
 10       34
 11       14
 12       31
 13       23
 14       15
 15       17
 16       22
 17       24
 18       27
 19       12
 20       12
 21       45
 22       21
 23       20
 24       11
 25        7
 26        9
Name: count, dtype: int64

In [28]:
pd.Series(train_cluster_labels).value_counts().sort_index()

-1      1239
 0     22707
 1      3423
 2       335
 3        25
 4       209
 5      1504
 6        89
 7        52
 8        53
 9        20
 10       34
 11       14
 12       31
 13       23
 14       15
 15       17
 16       22
 17       24
 18       27
 19       12
 20       12
 21       45
 22       21
 23       20
 24       11
 25        7
 26        9
Name: count, dtype: int64

## DBSCAN Training Cluster Analysis

DBSCAN was fitted on a sample of 30,000 BENIGN training flows using `eps=3.0` and `min_samples=10`.

The model assigned most samples to clusters and marked 1,239 samples as noise. This means approximately 4.13% of the normal training flows were considered low-density points.

The largest cluster contained 22,707 samples, while the second largest cluster contained 3,423 samples. Several smaller clusters were also found. This suggests that normal network traffic is not represented by a single homogeneous pattern, but instead contains multiple dense behavioral groups.

The presence of noise points among BENIGN data is expected because some normal flows may still be rare or unusual. In a monitoring system, these points could contribute to false positives if DBSCAN were used directly as an anomaly detector.

In [29]:
dbscan_param_results = []

for eps in [2.0, 2.5, 3.0, 3.5, 4.0, 5.0]:
    dbscan = DBSCAN(
        eps=eps,
        min_samples=10,
        n_jobs=-1
    )

    labels = dbscan.fit_predict(X_train_dbscan)

    num_noise = (labels == -1).sum()
    num_clusters = len(set(labels)) - (1 if -1 in labels else 0)

    dbscan_param_results.append({
        "eps": eps,
        "min_samples": 10,
        "num_clusters": num_clusters,
        "num_noise_points": num_noise,
        "noise_percentage": num_noise / len(labels)
    })

dbscan_param_results_df = pd.DataFrame(dbscan_param_results)
dbscan_param_results_df

,eps,min_samples,num_clusters,num_noise_points,noise_percentage
0,2.0,10,34,2233,0.074433
1,2.5,10,26,1661,0.055367
2,3.0,10,27,1239,0.041300
3,3.5,10,18,944,0.031467
4,4.0,10,20,770,0.025667
5,5.0,10,17,519,0.017300


## DBSCAN Parameter Sensitivity

The DBSCAN parameter comparison shows that the number of clusters and noise points depends strongly on the value of `eps`.

Smaller `eps` values create stricter neighborhoods. This leads to more fragmented clusters and a higher number of BENIGN samples being marked as noise. For example, with `eps = 2.0`, DBSCAN produced 34 clusters and marked 7.44% of the normal training samples as noise.

Larger `eps` values create more relaxed neighborhoods. This reduces the number of noise points and merges more samples into larger clusters. For example, with `eps = 5.0`, only 1.73% of the normal training samples were marked as noise.

The result shows that normal network traffic contains multiple dense behavioral groups rather than one single uniform pattern. The noise points represent rare or unusual BENIGN flows and could contribute to false positives if DBSCAN were used directly as an anomaly detector.

For this exploratory DBSCAN baseline, `eps = 3.0` is a reasonable middle setting because it produces a moderate number of clusters and marks around 4.13% of normal samples as noise.

## Autoencoder Anomaly Detection

An autoencoder is used as a neural anomaly detection model. The model is trained only on BENIGN network traffic and learns to reconstruct normal flow patterns.

During evaluation, the reconstruction error is calculated for each test sample. Samples with high reconstruction error are considered more anomalous. Since the model has only learned normal traffic behavior, DDoS flows are expected to produce higher reconstruction errors than BENIGN flows.

In [30]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [31]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [32]:
X_train_ae = X_train.sample(n=100000, random_state=42)

X_train_tensor = torch.tensor(X_train_ae.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)

print("Autoencoder train tensor:", X_train_tensor.shape)
print("Autoencoder test tensor:", X_test_tensor.shape)

Autoencoder train tensor: torch.Size([100000, 78])
Autoencoder test tensor: torch.Size([223082, 78])


In [33]:
input_dim = X_train.shape[1]

class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16)
        )

        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        reconstructed = self.decoder(encoded)
        return reconstructed

autoencoder = Autoencoder(input_dim).to(device)
autoencoder

Autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=78, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=16, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=16, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=78, bias=True)
  )
)

In [34]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)

num_epochs = 20

for epoch in range(num_epochs):
    autoencoder.train()
    total_loss = 0

    for batch in train_loader:
        x = batch[0].to(device)

        optimizer.zero_grad()
        reconstructed = autoencoder(x)
        loss = criterion(reconstructed, x)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.6f}")

D:\Development\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch [1/20], Loss: 0.598280
Epoch [2/20], Loss: 0.342343
Epoch [3/20], Loss: 0.270843
Epoch [4/20], Loss: 0.210473
Epoch [5/20], Loss: 0.176305
Epoch [6/20], Loss: 0.159154
Epoch [7/20], Loss: 0.161327
Epoch [8/20], Loss: 0.139169
Epoch [9/20], Loss: 0.106334
Epoch [10/20], Loss: 0.130848
Epoch [11/20], Loss: 0.108882
Epoch [12/20], Loss: 0.082430
Epoch [13/20], Loss: 0.107663
Epoch [14/20], Loss: 0.075743
Epoch [15/20], Loss: 0.074746
Epoch [16/20], Loss: 0.070911
Epoch [17/20], Loss: 0.106279
Epoch [18/20], Loss: 0.087496
Epoch [19/20], Loss: 0.057017
Epoch [20/20], Loss: 0.048351


In [35]:
autoencoder.eval()

with torch.no_grad():
    X_test_tensor_device = X_test_tensor.to(device)
    reconstructed_test = autoencoder(X_test_tensor_device)

    reconstruction_errors = torch.mean(
        (X_test_tensor_device - reconstructed_test) ** 2,
        dim=1
    ).cpu().numpy()

print("Reconstruction errors shape:", reconstruction_errors.shape)
print("Min error:", reconstruction_errors.min())
print("Max error:", reconstruction_errors.max())
print("Mean error:", reconstruction_errors.mean())

Reconstruction errors shape: (223082,)
Min error: 0.00077700056
Max error: 106.19377
Mean error: 4.5183706


In [36]:
from sklearn.metrics import roc_auc_score, average_precision_score

ae_roc_auc = roc_auc_score(y_test, reconstruction_errors)
ae_pr_auc = average_precision_score(y_test, reconstruction_errors)

print("Autoencoder ROC-AUC:", ae_roc_auc)
print("Autoencoder PR-AUC:", ae_pr_auc)

Autoencoder ROC-AUC: 0.7532766656511078
Autoencoder PR-AUC: 0.7911662483092397


In [37]:
ae_score_df = pd.DataFrame({
    "label": y_test_labels,
    "binary_label": y_test,
    "reconstruction_error": reconstruction_errors
})

ae_score_df.groupby("label")["reconstruction_error"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
BENIGN,95068.0,1.527243,4.893819,0.000777,0.002429,0.036521,0.516929,106.193771
DDoS,128014.0,6.739694,11.987817,0.010369,0.055667,1.991383,7.049413,93.008072


## Autoencoder Result Interpretation

The autoencoder achieved a ROC-AUC of approximately 0.75 and a PR-AUC of approximately 0.79. This shows that reconstruction error provides a useful anomaly signal for distinguishing DDoS traffic from BENIGN traffic.

The reconstruction error distribution shows that DDoS flows have a higher mean and median reconstruction error than BENIGN flows. This indicates that the autoencoder, trained only on BENIGN traffic, reconstructs normal traffic more accurately than DDoS traffic.

The final training loss decreased to approximately 0.048, showing that the model learned to reconstruct the normal training data. Some loss fluctuations were observed during training, which can be improved in future iterations using validation-based early stopping, architecture tuning, or training on a larger normal sample.

Overall, the autoencoder provides a stronger baseline than Isolation Forest and gives an interpretable anomaly score based on reconstruction error.

In [38]:
autoencoder.eval()

with torch.no_grad():
    X_train_tensor_device = X_train_tensor.to(device)
    reconstructed_train = autoencoder(X_train_tensor_device)

    train_reconstruction_errors = torch.mean(
        (X_train_tensor_device - reconstructed_train) ** 2,
        dim=1
    ).cpu().numpy()

print("Train reconstruction errors shape:", train_reconstruction_errors.shape)
print("Min:", train_reconstruction_errors.min())
print("Max:", train_reconstruction_errors.max())
print("Mean:", train_reconstruction_errors.mean())

Train reconstruction errors shape: (100000,)
Min: 0.00075711036
Max: 388.2159
Mean: 0.071123995


In [39]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

ae_threshold_results = []

for percentile in [50, 60, 70, 80, 85, 90, 95, 97, 99]:
    threshold = np.percentile(train_reconstruction_errors, percentile)
    y_pred = (reconstruction_errors > threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    fpr = fp / (fp + tn)

    ae_threshold_results.append({
        "percentile": percentile,
        "threshold": threshold,
        "precision": precision,
        "recall_detection_rate": recall,
        "f1_score": f1,
        "false_positive_rate": fpr,
        "predicted_anomalies": y_pred.sum()
    })

ae_threshold_results_df = pd.DataFrame(ae_threshold_results)
ae_threshold_results_df

,percentile,threshold,precision,recall_detection_rate,f1_score,false_positive_rate,predicted_anomalies
0,50,0.008879,0.673868,1.000000,0.805163,0.651691,189969
1,60,0.013407,0.679392,0.918884,0.781195,0.583898,173140
2,70,0.026798,0.680044,0.854164,0.757224,0.541149,160791
3,80,0.044696,0.693766,0.787945,0.737862,0.468338,145392
4,85,0.063714,0.693118,0.726249,0.709297,0.432985,134133
5,90,0.107847,0.682860,0.640149,0.660815,0.400334,120007
6,95,0.238195,0.701725,0.636016,0.667257,0.364034,116027
7,97,0.405250,0.754599,0.627752,0.685355,0.274898,106495
8,99,0.898067,0.804997,0.587904,0.679533,0.191768,93491


## Autoencoder Threshold Comparison Interpretation

The autoencoder threshold comparison shows the trade-off between DDoS detection rate and false positive rate. Lower reconstruction-error thresholds detect more DDoS flows, but they also incorrectly flag many BENIGN flows as anomalous.

At the 50th percentile threshold, the autoencoder detects nearly all DDoS traffic, but the false positive rate is approximately 65%, which would be too noisy for a practical monitoring system.

Higher thresholds reduce false positives and improve precision. The 97th percentile threshold provides a practical balance, achieving approximately 75% precision, 63% recall, and an F1-score of about 0.69, with a false positive rate of around 27%.

The 99th percentile threshold gives the lowest false positive rate among the tested autoencoder thresholds, but it also reduces the detection rate. Therefore, the 97th percentile threshold is selected as the balanced autoencoder operating point for this experiment.

In [40]:
final_model_comparison = pd.DataFrame([
    {
        "model": "Isolation Forest",
        "selected_threshold_percentile": 90,
        "roc_auc": 0.680898,
        "pr_auc": 0.649548,
        "precision": 0.770613,
        "recall": 0.636071,
        "f1_score": 0.696908,
        "false_positive_rate": 0.254954,
        "notes": "Tree-based baseline"
    },
    {
        "model": "Local Outlier Factor",
        "selected_threshold_percentile": 95,
        "roc_auc": 0.802653,
        "pr_auc": 0.811594,
        "precision": 0.789067,
        "recall": 0.573687,
        "f1_score": 0.664357,
        "false_positive_rate": 0.206505,
        "notes": "Best ranking performance"
    },
    {
        "model": "Autoencoder",
        "selected_threshold_percentile": 97,
        "roc_auc": 0.753277,
        "pr_auc": 0.791166,
        "precision": 0.754599,
        "recall": 0.627752,
        "f1_score": 0.685355,
        "false_positive_rate": 0.274898,
        "notes": "Neural reconstruction-based detector"
    },
    {
        "model": "DBSCAN",
        "selected_threshold_percentile": None,
        "roc_auc": None,
        "pr_auc": None,
        "precision": None,
        "recall": None,
        "f1_score": None,
        "false_positive_rate": None,
        "notes": "Exploratory clustering baseline; no direct novelty scoring"
    }
])

final_model_comparison

,model,selected_threshold_percentile,roc_auc,pr_auc,precision,recall,f1_score,false_positive_rate,notes
0,Isolation Forest,90.0,0.680898,0.649548,0.770613,0.636071,0.696908,0.254954,Tree-based baseline
1,Local Outlier Factor,95.0,0.802653,0.811594,0.789067,0.573687,0.664357,0.206505,Best ranking performance
2,Autoencoder,97.0,0.753277,0.791166,0.754599,0.627752,0.685355,0.274898,Neural reconstruction-based detector
3,DBSCAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Exploratory clustering baseline; no direct nov...


## Final Baseline Model Comparison

The final comparison shows that all three main anomaly detection models were able to detect DDoS traffic to some extent when trained only on BENIGN traffic.

Local Outlier Factor achieved the best ranking performance, with the highest ROC-AUC and PR-AUC. This indicates that LOF was most effective at assigning higher anomaly scores to DDoS flows compared to BENIGN flows.

The autoencoder also performed well and produced a meaningful reconstruction-error signal. DDoS flows had higher reconstruction errors than BENIGN flows, showing that the model learned normal traffic patterns and struggled more to reconstruct attack traffic.

Isolation Forest provided a useful initial baseline, but its ranking performance was weaker than LOF and the autoencoder.

DBSCAN was used as an exploratory density-based clustering baseline. It showed that normal traffic forms multiple dense clusters, but it was not used as a full test-set anomaly detector because scikit-learn DBSCAN does not provide direct novelty scoring for unseen test samples.

Overall, LOF and the autoencoder are the strongest models in this first experiment. LOF provides the best ranking performance, while the autoencoder provides an interpretable neural reconstruction-based anomaly score. Further improvements could include feature selection, dimensionality reduction, hyperparameter tuning, and evaluation on additional attack types.